In [0]:
-----------------------------------------------------------------------------
-- BRIGHT COFFEE SHOP SALES ANALYSIS CASE STUDY
-----------------------------------------------------------------------------

-- 0. PREVIEW DATA
SELECT *
FROM workspace.default.bright_coffee_shop_analysis_case_study_1_2
LIMIT 10;


-----------------------------------------------------------------------------
-- 1. CHECKING THE DATE RANGE
----------------------------------------------------------------------------

SELECT MIN(transaction_date) AS min_date
FROM workspace.default.bright_coffee_shop_analysis_case_study_1_2;


SELECT MAX(transaction_date) AS latest_date
FROM workspace.default.bright_coffee_shop_analysis_case_study_1_2;


-----------------------------------------------------------------------------
-- 2. STORE ANALYSIS
-----------------------------------------------------------------------------

SELECT DISTINCT store_location
FROM workspace.default.bright_coffee_shop_analysis_case_study_1_2;


SELECT COUNT(DISTINCT store_id) AS number_of_stores
FROM workspace.default.bright_coffee_shop_analysis_case_study_1_2;


-----------------------------------------------------------------------------
-- 3. PRODUCT ANALYSIS
-----------------------------------------------------------------------------

SELECT DISTINCT product_category
FROM workspace.default.bright_coffee_shop_analysis_case_study_1_2;


SELECT DISTINCT product_detail
FROM workspace.default.bright_coffee_shop_analysis_case_study_1_2;


SELECT DISTINCT product_type
FROM workspace.default.bright_coffee_shop_analysis_case_study_1_2;


SELECT DISTINCT 
    product_category AS category,
    product_detail AS product_name
FROM workspace.default.bright_coffee_shop_analysis_case_study_1_2;


-----------------------------------------------------------------------------
-- 4. PRICE ANALYSIS
-----------------------------------------------------------------------------

SELECT MIN(unit_price) AS cheapest_price
FROM workspace.default.bright_coffee_shop_analysis_case_study_1_2;


SELECT MAX(unit_price) AS expensive_price
FROM workspace.default.bright_coffee_shop_analysis_case_study_1_2;


-----------------------------------------------------------------------------
-- 5. COUNT IDS
-----------------------------------------------------------------------------

SELECT
COUNT(*) AS total_rows,
COUNT(DISTINCT transaction_id) AS number_of_sales,
COUNT(DISTINCT product_id) AS number_of_products,
COUNT(DISTINCT store_id) AS number_of_stores
FROM workspace.default.bright_coffee_shop_analysis_case_study_1_2;


-----------------------------------------------------------------------------
-- 6.CALCULATIONS
-----------------------------------------------------------------------------

SELECT
transaction_id,
transaction_date,
DAYNAME(transaction_date) AS day_name,
MONTHNAME(transaction_date) AS month_name,
transaction_qty * unit_price AS revenue_per_transaction
FROM workspace.default.bright_coffee_shop_analysis_case_study_1_2;


-----------------------------------------------------------------------------
-- 7. DATE & TIME ANALYSIS
-----------------------------------------------------------------------------

SELECT
transaction_date,
DAYNAME(transaction_date) AS day_name,
MONTHNAME(transaction_date) AS month_name,
DATE_FORMAT(transaction_time, 'HH:mm:ss') AS purchase_time
FROM workspace.default.bright_coffee_shop_analysis_case_study_1_2
GROUP BY 
transaction_date,
day_name,
month_name,
DATE_FORMAT(transaction_time, 'HH:mm:ss');


-----------------------------------------------------------------------------
-- 8. FULL ANALYSIS QUERY
-----------------------------------------------------------------------------

SELECT
transaction_date AS purchase_date,
DAYNAME(transaction_date) AS day_name,
MONTHNAME(transaction_date) AS month_name,
DAYOFMONTH(transaction_date) AS day_of_month,

CASE
WHEN DAYNAME(transaction_date) IN ('Monday','Tuesday', 'Wednesday','Thursday','Friday','Saturday', 'Sunday') THEN 'All Days'
END AS day_classification,

DATE_FORMAT(transaction_time, 'HH:mm:ss') AS purchase_time,

CASE
WHEN DATE_FORMAT(transaction_time, 'HH:mm:ss') BETWEEN '00:00:00' AND '11:59:59' THEN '01. Morning'
WHEN DATE_FORMAT(transaction_time, 'HH:mm:ss') BETWEEN '12:00:00' AND '16:59:59' THEN '02. Afternoon'
WHEN DATE_FORMAT(transaction_time, 'HH:mm:ss') BETWEEN '17:00:00' AND '23:59:59' THEN '03. Evening'
END AS time_buckets,

COUNT(DISTINCT transaction_id) AS number_of_sales,
COUNT(DISTINCT product_id) AS number_of_products,
COUNT(DISTINCT store_id) AS number_of_stores,

SUM(transaction_qty * unit_price) AS revenue_per_day,

CASE
WHEN SUM(transaction_qty * unit_price) <= 50 THEN '01. Low spend'
WHEN SUM(transaction_qty * unit_price) BETWEEN 51 AND 100 THEN '02. Medium spend'
ELSE '03. High spend'
END AS revenue_buckets,

store_location,
product_category,
product_detail

FROM workspace.default.bright_coffee_shop_analysis_case_study_1_2

GROUP BY 
transaction_date,
DAYNAME(transaction_date),
MONTHNAME(transaction_date),
DAYOFMONTH(transaction_date),
DATE_FORMAT(transaction_time, 'HH:mm:ss'),
store_location,
product_category,
product_detail;

-----------------------------------------------------------------------------
-- 9. SALES BY STORE 
-----------------------------------------------------------------------------

SELECT
store_location,
COUNT(DISTINCT transaction_id) AS total_sales,
SUM(transaction_qty * unit_price) AS total_revenue
FROM workspace.default.bright_coffee_shop_analysis_case_study_1_2
GROUP BY store_location
ORDER BY total_revenue DESC;

-----------------------------------------------------------------------------
-- 10. SALES BY PRODUCT CATEGORY
-----------------------------------------------------------------------------

SELECT
product_category,
COUNT(DISTINCT transaction_id) AS total_sales,
SUM(transaction_qty * unit_price) AS total_revenue
FROM workspace.default.bright_coffee_shop_analysis_case_study_1_2
GROUP BY product_category
ORDER BY total_revenue DESC;



------------------------------------------------------------------------
SELECT

-- DATE STRUCTURE
transaction_date AS purchase_date,
DAYNAME(transaction_date) AS day_name,
MONTHNAME(transaction_date) AS month_name,
DAYOFMONTH(transaction_date) AS day_of_month,

-- DAY NUMBER (FOR CORRECT SORTING MON → SUN)
((DAYOFWEEK(transaction_date) + 5) % 7) + 1 AS day_number,

-- WEEK TYPE
CASE
    WHEN DAYOFWEEK(transaction_date) IN (1,7) THEN 'Weekend'
    ELSE 'Weekday'
END AS day_type,

-- CLEAN TIME
CAST(transaction_time AS TIMESTAMP) AS full_timestamp,
DATE_FORMAT(transaction_time, 'HH:mm:ss') AS purchase_time,

-- HOUR (FOR DEEP ANALYSIS)
HOUR(CAST(transaction_time AS TIMESTAMP)) AS hour,

-- TIME OF DAY (MAIN GROUPING)
CASE
    WHEN HOUR(CAST(transaction_time AS TIMESTAMP)) < 12 THEN '01. Morning'
    WHEN HOUR(CAST(transaction_time AS TIMESTAMP)) < 17 THEN '02. Afternoon'
    WHEN HOUR(CAST(transaction_time AS TIMESTAMP)) < 21 THEN '03. Evening'
    ELSE '04. Night'
END AS time_of_day,

-- TIME BUCKET (HOURLY GROUPING)
DATE_FORMAT(CAST(transaction_time AS TIMESTAMP), 'HH:00') AS time_bucket,

-- CORE METRICS
COUNT(DISTINCT transaction_id) AS number_of_sales,
COUNT(DISTINCT product_id) AS number_of_products,
COUNT(DISTINCT store_id) AS number_of_stores,

SUM(transaction_qty * unit_price) AS total_revenue,

-- REVENUE SEGMENTATION
CASE
    WHEN SUM(transaction_qty * unit_price) <= 50 THEN '01. Low spend'
    WHEN SUM(transaction_qty * unit_price) BETWEEN 51 AND 100 THEN '02. Medium spend'
    ELSE '03. High spend'
END AS revenue_bucket,

-- DIMENSIONS
store_location,
product_category,
product_detail

FROM workspace.default.bright_coffee_shop_analysis_case_study_1_2

GROUP BY 
transaction_date,
DAYNAME(transaction_date),
MONTHNAME(transaction_date),
DAYOFMONTH(transaction_date),
((DAYOFWEEK(transaction_date) + 5) % 7) + 1,
CASE WHEN DAYOFWEEK(transaction_date) IN (1,7) THEN 'Weekend' ELSE 'Weekday' END,
CAST(transaction_time AS TIMESTAMP),
DATE_FORMAT(transaction_time, 'HH:mm:ss'),
HOUR(CAST(transaction_time AS TIMESTAMP)),
DATE_FORMAT(CAST(transaction_time AS TIMESTAMP), 'HH:00'),
store_location,
product_category,
product_detail;
------------------------------------------------------------------------------------
--------------------------------------------------------------------------------
SELECT

-- ORIGINAL FIELDS
transaction_id,
transaction_date,
transaction_time,
store_location,
product_category,
product_detail,
transaction_qty,
unit_price,


FROM workspace.default.bright_coffee_shop_analysis_case_study_1_2
LIMIT 1000000;